# 5. Topic modeling: LDA (Section 4.4, Tables 4-7)

Gensim `LdaMulticore` (passes=5, workers=4). The original run had no random seed; here `random_state=42` is fixed, so values can differ slightly from the paper. Coherence uses gensim's default top 20 words, as in the paper run.

The number of topics k is swept over 2, 4, ..., 28 and the model with the highest C_V coherence is
selected. All metrics below are reported for that selected model.
See docs/REPRODUCIBILITY_NOTES.md, section 5, about the C_NPMI values in the paper.

In [ ]:
from utils import *
from gensim import corpora
from gensim.models import LdaMulticore, CoherenceModel

In [ ]:
summary = {}
for platform in ["whatsapp", "telegram"]:
    texts = tokenize_for_topics(load_dataset(platform)["text_processed"])
    dictionary = corpora.Dictionary(texts)
    corpus = [dictionary.doc2bow(t) for t in texts]

    sweep, models = [], {}
    for k in range(2, 30, 2):
        model = LdaMulticore(corpus=corpus, id2word=dictionary, num_topics=k,
                             passes=5, workers=4, random_state=SEED)
        coherence = {m: CoherenceModel(model=model, texts=texts, dictionary=dictionary,
                                       coherence=m).get_coherence() for m in ["c_v", "c_npmi"]}
        sweep.append({"k": k, **coherence}); models[k] = model

    sweep = pd.DataFrame(sweep)
    best = sweep.loc[sweep["c_v"].idxmax()]
    print(sweep.round(4).to_string(index=False))
    print("\nselected k =", int(best["k"]))
    model = models[int(best["k"])]
    topics = [[w for w, _ in model.show_topic(i, topn=10)] for i in range(model.num_topics)]
    summary[NAMES[platform]] = {"k": int(best["k"]), "c_npmi": best["c_npmi"], "c_v": best["c_v"],
                                "irbo": irbo(topics), "diversity": topic_diversity(topics)}

    out = results_folder("05_topics_lda")
    sweep.to_csv(out / f"lda_{NAMES[platform]}_k_sweep.csv", index=False)
    pd.DataFrame({"topic": range(len(topics)), "words": ["; ".join(t) for t in topics]}) \
      .to_csv(out / f"lda_{NAMES[platform]}_topics.csv", index=False)

summary = pd.DataFrame(summary).round(3)
summary

## Comparison with the paper (Tables 4 and 5)

In [ ]:
compare_with_paper([
    ("FW.Br k", 6, summary.loc["k", "FW.Br"]), ("FW.Br C_V", 0.530, summary.loc["c_v", "FW.Br"]),
    ("FW.Br IRBO", 0.41, summary.loc["irbo", "FW.Br"]), ("FW.Br diversity", 0.48, summary.loc["diversity", "FW.Br"]),
    ("FW.Br C_NPMI (paper value is from k=2, see notes)", 0.002, summary.loc["c_npmi", "FW.Br"]),
    ("FT.Br k", 2, summary.loc["k", "FT.Br"]), ("FT.Br C_V", 0.514, summary.loc["c_v", "FT.Br"]),
    ("FT.Br IRBO", 0.58, summary.loc["irbo", "FT.Br"]), ("FT.Br diversity", 0.80, summary.loc["diversity", "FT.Br"]),
    ("FT.Br C_NPMI (paper value is from k=20, see notes)", -0.008, summary.loc["c_npmi", "FT.Br"]),
], tolerance=0.01)